## Race Position Timeline

In [54]:
import pandas as pd
import numpy as np
import altair as alt
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)

# Ensure Altair allows larger datasets
alt.data_transformers.enable("default", max_rows = None)

lap_and_pit_times_df = pd.read_csv("../../data/lap_and_pit_times.csv")

C:\Users\DCorc\AppData\Local\Temp\ipykernel_7196\930004920.py:12: DtypeWarning: Columns (14,15) have mixed types. Specify dtype option on import or set low_memory=False.
  lap_and_pit_times_df = pd.read_csv("../../data/lap_and_pit_times.csv")


In [55]:
# Calculate change in position after pitstops
# Ensure sorting for position tracking
pitstop_effect_df = lap_and_pit_times_df.sort_values(by = ["raceId", "driverId", "lap"])

# Get position before pit stop (previous lap)
#pitstop_effect_df["position_before"] = pitstop_effect_df.groupby(["raceId", "driverId"])["position"].shift(1)

# Get position after pit stop (next lap)
pitstop_effect_df["position_after"] = pitstop_effect_df.groupby(["raceId", "driverId"])["position"].shift(-1)

# Calculate position change
pitstop_effect_df["position_change"] = pitstop_effect_df["position"] - pitstop_effect_df["position_after"]

# Convert milliseconds to seconds
pitstop_effect_df["pitStopSeconds"] = pitstop_effect_df["pitStopMilliseconds"] / 1000

pitstop_effect_df = pitstop_effect_df.dropna(subset = ["pitStopSeconds"])

# Filter only rows with actual pit stops
pitstop_df = pitstop_effect_df[pitstop_effect_df["pitstop"] == True].dropna()

In [ ]:
# Ensure Altair allows larger datasets
alt.data_transformers.enable("default", max_rows = None)

# Dropdown for Year
year_dropdown = alt.binding_select(options = sorted(lap_and_pit_times_df["year"].dropna().unique()), name = "Year: ")
year_selection = alt.selection_single(bind = year_dropdown, fields=["year"], name = "year_selection", value = 2024)

# Dropdown for Circuit
circuit_dropdown = alt.binding_select(options = sorted(lap_and_pit_times_df["circuitName"].dropna().unique()), name = "Circuit: ")
circuit_selection = alt.selection_single(bind = circuit_dropdown, fields = ["circuitName"], name = "circuit_selection", value = "United States Grand Prix")

# Create a multi-selection for drivers to toggle visibility
driver_selection = alt.selection_multi(fields = ["surname"], bind = "legend", name = "driver_selection", empty = "all")

# Dropdown for Constructor
constructor_dropdown = alt.binding_select(options = sorted(lap_and_pit_times_df["constructorName"].dropna().unique()), name = "Team: ")
constructor_selection = alt.selection_single(bind = constructor_dropdown, fields = ["constructorName"], name = "constructor_selection", value = "Red Bull")

max_position = lap_and_pit_times_df["position"].max()

# Base chart: Driver's position over laps
base_chart = alt.Chart(lap_and_pit_times_df).mark_line().encode(
    x = alt.X(
        "lap:O", 
        title = "Lap Number"
    ),
    y = alt.Y(
        "position:Q", 
        title = "Race Position", 
        sort = "descending",
        scale = alt.Scale(domain = [1, max_position]),
        axis = alt.Axis(values = list(range(1, max_position + 1)))
    ),
    color = alt.condition(
        driver_selection,
        alt.Color("surname:N", legend = alt.Legend(title = "Driver Name", orient = "right"), scale = alt.Scale(range = ["#15151E", "#FF1E00"])),
        alt.value("lightgray")
    ),
    order = alt.condition(
        driver_selection,
        alt.value(3),
        alt.value(0)
    ),
    size = alt.condition(
        driver_selection,
        alt.value(5),
        alt.value(2)
    ),
    tooltip = [
        alt.Tooltip("forename:N", title = "Driver First Name"),
        alt.Tooltip("surname:N", title = "Driver Last Name"),
        alt.Tooltip("lap:O", title = "Lap Number"),
        alt.Tooltip("position:Q", title = "Current Position"),
        alt.Tooltip("constructorName:Q", title = "Constructor")
    ]
).transform_filter(
    year_selection
).transform_filter(
    circuit_selection
).transform_filter(
    constructor_selection
).properties(
    title = "F1 Driver Position Timeline",
    width = 450,
    height = 300
)

# Add pit stop data as points 
pit_stops = alt.Chart(lap_and_pit_times_df).transform_calculate(
    pitstop_event = "'Pit Stop'"
).mark_point(
    shape = "circle",
    size = 200,
    filled = True,
    stroke = "black",
    strokeWidth = 1.5,
    opacity = 1
).encode(
    x = "lap:O",
    y = alt.Y("position:Q",
              scale = alt.Scale(domain = [1, max_position]),
                axis = alt.Axis(values = list(range(1, max_position + 1)))
    ),
    shape = alt.Shape("pitstop_event:N", legend = alt.Legend(title = "Event")
    ),
    color = alt.value("#FFD700"),
    tooltip = [
        alt.Tooltip("forename:N", title = "Driver First Name"),
        alt.Tooltip("surname:N", title = "Driver Last Name"),
        alt.Tooltip("lap:O", title = "Lap Number"),
        alt.Tooltip("pitStopDuration:Q", title = "Pit Stop Duration")
    ]
).transform_filter(
    year_selection
).transform_filter(
    circuit_selection
).transform_filter(
    constructor_selection
).transform_filter(
    alt.datum.pitstop == True
).transform_filter(
    driver_selection
)

# Combine the base chart with the pit stop points
interactive_chart = alt.layer(
    base_chart,
    pit_stops
).add_params(
    year_selection, circuit_selection, driver_selection, constructor_selection
)

# Set the configurations
interactive_chart = interactive_chart.configure(
    title = {"font": "Quicksand, sans-serif", "fontSize": 20},
    axis = {"labelFont": "Quicksand, sans-serif", "titleFont": "Quicksand, sans-serif", "labelFontSize": 14, "titleFontSize": 16},
    legend = {"labelFont": "Quicksand, sans-serif", "titleFont": "Quicksand, sans-serif", "labelFontSize": 14, "titleFontSize": 16}
)

#interactive_chart

#interactive_chart.save("../../img/driver_position_timeline.html")

C:\Users\DCorc\AppData\Local\Temp\ipykernel_7196\2901774104.py:6: AltairDeprecationWarning: 
Deprecated since `altair=5.0.0`. Use selection_point instead.
  year_selection = alt.selection_single(bind = year_dropdown, fields=["year"], name = "year_selection", value = 2024)
C:\Users\DCorc\AppData\Local\Temp\ipykernel_7196\2901774104.py:10: AltairDeprecationWarning: 
Deprecated since `altair=5.0.0`. Use selection_point instead.
  circuit_selection = alt.selection_single(bind = circuit_dropdown, fields = ["circuitName"], name = "circuit_selection", value = "United States Grand Prix")
C:\Users\DCorc\AppData\Local\Temp\ipykernel_7196\2901774104.py:13: AltairDeprecationWarning: 
Deprecated since `altair=5.0.0`. Use selection_point instead.
  driver_selection = alt.selection_multi(fields = ["surname"], bind = "legend", name = "driver_selection", empty = "all")
C:\Users\DCorc\AppData\Local\Temp\ipykernel_7196\2901774104.py:17: AltairDeprecationWarning: 
Deprecated since `altair=5.0.0`. Use sele

## Pit Stop Duration vs. Position Change (Scatter Plot)

In [ ]:
# Ensure sorting for position tracking
pitstop_df = lap_and_pit_times_df.sort_values(by = ["raceId", "driverId", "lap"])

# Get position before pit stop (previous lap)
pitstop_df["position_before"] = pitstop_df.groupby(["raceId", "driverId"])["position"].shift(1)

# Calculate position change (before - after pit stop)
pitstop_df["position_change"] = pitstop_df["position_before"] - pitstop_df["position"]

# Convert milliseconds to seconds
pitstop_df["pitStopSeconds"] = pitstop_df["pitStopMilliseconds"] / 1000

pitstop_df = pitstop_df.dropna(subset=["pitStopSeconds"])

# Filter only rows with actual pit stops
pitstop_df = pitstop_df[pitstop_df["pitstop"] == True].dropna()

# Outlier Removal using IQR
Q1 = pitstop_df["pitStopSeconds"].quantile(0.25)
Q3 = pitstop_df["pitStopSeconds"].quantile(0.75)
IQR = Q3 - Q1

# Define lower and upper bounds
upper_bound = Q3 + 1.5 * IQR

# Filter out outliers
pitstop_df = pitstop_df[pitstop_df["pitStopSeconds"] <= upper_bound]

'''
# Scatter plot: Pit Stop Duration vs. Position Change
plt.figure(figsize = (10, 6))
sns.scatterplot(x = pitstop_df["pitStopSeconds"], y = pitstop_df["position_change"], alpha = 0.7)
plt.xlabel("Pit Stop Duration (Seconds)")
plt.ylabel("Position Change (Before - After)")
plt.title("Impact of Pit Stop Duration on Position Change")
plt.axhline(0, color = "gray", linestyle = "dashed")

#plt.savefig("../../img/pit_stop_duration_on_position.png", format="png")
plt.show()
'''

## Pit Stop Timing vs Outcome (Box Plots)

In [ ]:
# Ensure pitStopMilliseconds is numeric
lap_and_pit_times_df["pitStopMilliseconds"] = pd.to_numeric(lap_and_pit_times_df["pitStopMilliseconds"], errors = "coerce")

# Filter only pit stop events
pitstop_df = lap_and_pit_times_df[lap_and_pit_times_df["pitstop"] == True]

# First & Last Pit Stop Lap per Race/Driver
first_pitstop = pitstop_df.groupby(["raceId", "driverId"])["lap"].min().reset_index()
last_pitstop = pitstop_df.groupby(["raceId", "driverId"])["lap"].max().reset_index()

# Rename columns
first_pitstop.rename(columns = {"lap": "first_pit_lap"}, inplace = True)
last_pitstop.rename(columns = {"lap": "last_pit_lap"}, inplace = True)

# Get Final Race Position
final_positions = lap_and_pit_times_df.groupby(["raceId", "driverId"])["position"].last().reset_index()
final_positions.rename(columns = {"position": "final_position"}, inplace = True)

# Merge Data
pitstop_analysis_df = first_pitstop.merge(last_pitstop, on = ["raceId", "driverId"])
pitstop_analysis_df = pitstop_analysis_df.merge(final_positions, on = ["raceId", "driverId"])

# Convert final position to integer
pitstop_analysis_df["final_position"] = pitstop_analysis_df["final_position"].astype(int)

# Box Plot - First Pit Stop vs Final Position
'''
plt.figure(figsize = (15, 6))
sns.boxplot(x = pitstop_analysis_df["first_pit_lap"], y = pitstop_analysis_df["final_position"], palette = "coolwarm")
plt.gca().invert_yaxis()
plt.xlabel("Lap of First Pit Stop")
plt.ylabel("Final Race Position")
plt.title("Impact of First Pit Stop Timing on Final Position")
plt.xticks(rotation=90)
#plt.savefig("../../img/first_pit_stop_timing_on_position.png", format="png")


# Box Plot - Last Pit Stop vs Final Position
plt.figure(figsize = (15, 6))
sns.boxplot(x = pitstop_analysis_df["last_pit_lap"], y = pitstop_analysis_df["final_position"], palette = "coolwarm")
plt.gca().invert_yaxis()
plt.xlabel("Lap of Last Pit Stop")
plt.ylabel("Final Race Position")
plt.title("Impact of Last Pit Stop Timing on Final Position")
plt.xticks(rotation = 90)
plt.savefig("../../img/last_pit_stop_timing_on_position.png", format="png")
plt.show()
'''

## Pit Stop Duration vs Final Race Position

In [ ]:
# Get Final Race Position
final_positions = lap_and_pit_times_df.groupby(["raceId", "driverId"])["position"].last().reset_index()
final_positions.rename(columns = {"position": "final_position"}, inplace = True)

# Group by raceId and driverId, summing the pit stop durations
grouped_df = lap_and_pit_times_df.groupby(["raceId", "driverId", "forename", "surname", "nationality", "circuitName", "year"], as_index = False).agg({
    "pitStopDuration": "sum"
}).merge(final_positions, on = ["raceId", "driverId"])

grouped_df["pitStopDuration"] = pd.to_numeric(grouped_df["pitStopDuration"], errors = "coerce")
grouped_df = grouped_df.dropna(subset = ["pitStopDuration"])
grouped_df = grouped_df[grouped_df["pitStopDuration"] > 0]



# Create a selection filter for the year
year_selector = alt.selection_single(
    fields = ["year"],
    name = "Year Select",
    bind = alt.binding_select(options = sorted(grouped_df["year"].unique()), name = "Year"),
    value = {"year": 2024} 
)

# Create the scatter plot
pit_stop_duration_scatter_plot = alt.Chart(grouped_df).mark_circle(size = 60).encode(
    x = alt.X("pitStopDuration:Q", 
              title = "Total Pit Stop Duration (Seconds)"),
    y = alt.Y("final_position:Q", 
              title = "Final Position", 
              scale = alt.Scale(reverse = True, domain = [1, grouped_df["final_position"].max()]),
              axis = alt.Axis(title = "Final Position", tickCount = grouped_df["final_position"].max())),
    color = alt.Color("surname:N", 
                      title = "Driver Name",),
    tooltip = [
        alt.Tooltip("forename", title = "Driver First Name"),
        alt.Tooltip("surname", title = "Driver Last Name"),
        alt.Tooltip("nationality", title = "Driver Nationality"),
        alt.Tooltip("pitStopDuration", title = "Total Pit Stop Duration (Seconds)"),
        alt.Tooltip("final_position", title = "Final Race Position"),
        alt.Tooltip("circuitName", title = "Circuit Name")
    ]
).add_selection(
    year_selector
).transform_filter(
    year_selector
).properties(
    title = "Pit Stop Duration vs Final Position",
    width = 600,
    height = 400
)

# Display the plot
#pit_stop_duration_scatter_plot

#pit_stop_duration_scatter_plot.save("../../img/pit_stop_duration_scatter_plot.html")

## Pit Stop Distributions

In [58]:
import pandas as pd

# Load tables
results_df = pd.read_csv("../../data/ergast/results.csv")         
constructors_df = pd.read_csv("../../data/ergast/constructors.csv") 
races_df = pd.read_csv("../../data/ergast/races.csv")       

# Merge results with constructors on constructorId
results_with_constructors_df = pd.merge(results_df, constructors_df, on = "constructorId", how = "inner")

# Merge with races to add the 'year' column based on raceId
results_with_constructors_and_year_df = pd.merge(results_with_constructors_df, races_df[["raceId", "year"]], on = "raceId", how = "inner")

# Convert position to numeric (invalid entries become NaN)
results_with_constructors_and_year_df["position"] = pd.to_numeric(results_with_constructors_and_year_df["position"], errors = "coerce")

# Filter top 3 finishes
top_3_finishes_df = results_with_constructors_and_year_df[results_with_constructors_and_year_df["position"].isin([1, 2, 3])]

# Group by constructor name and year, and count the number of top 3 finishes per year
constructor_top_3_per_year_df = top_3_finishes_df.groupby(["name", "year"], as_index = False).size()

# Rename columns for clarity
constructor_top_3_per_year_df.columns = ["constructorName", "year", "Top 3 Finishes"]

# Sort by year and top 3 finishes descending
constructor_top_3_per_year_df = constructor_top_3_per_year_df.sort_values(["year", "Top 3 Finishes"], ascending = [True, False])
constructor_top_3_per_year_df = constructor_top_3_per_year_df[constructor_top_3_per_year_df["year"] >= 2009]

# Show the result
print(constructor_top_3_per_year_df)

print(constructor_top_3_per_year_df.dtypes)

    constructorName  year  Top 3 Finishes
346        Red Bull  2009              16
82            Brawn  2009              15
161         Ferrari  2009               6
305         McLaren  2009               5
421          Toyota  2009               5
..              ...   ...             ...
176         Ferrari  2024              22
315         McLaren  2024              21
361        Red Bull  2024              18
334        Mercedes  2024               9
10   Alpine F1 Team  2024               2

[88 rows x 3 columns]
constructorName    object
year                int64
Top 3 Finishes      int64
dtype: object


In [60]:
import altair as alt

# Dropdown for Constructor
constructor_dropdown = alt.binding_select(
    options = sorted(lap_and_pit_times_df["constructorName"].dropna().unique()), 
    name = "Team: "
)
constructor_selection = alt.selection_single(
    bind = constructor_dropdown, 
    fields = ["constructorName"], 
    name = "constructor_selection", 
    value = "Red Bull"
)

# Dropdown for Year
year_dropdown = alt.binding_select(
    options = sorted(lap_and_pit_times_df["year"].dropna().unique()), 
    name = "Year: "
)
year_selection = alt.selection_single(
    bind = year_dropdown, 
    fields = ["year"], 
    name = "year_selection", 
    value = 2024
)

# Create a histogram for pit stop times
pit_stop_histogram = alt.Chart(lap_and_pit_times_df).mark_bar(
    stroke = "#15151E",
    strokeWidth = 1
).encode(
    alt.X(
        "pitStopDuration:Q",
        bin = alt.Bin(extent = [10, 60], maxbins = 50),
        title = "Pit Stop Duration (Seconds)",
        scale = alt.Scale(domain = [10, 60]),
    ),
    alt.Y(
        "count():Q",
        title = "Count of Pit Stops",
        scale = alt.Scale(domain = [0, 22])
    ),
    color = alt.value("#FF1E00"),
    tooltip = [
        alt.Tooltip("count():Q", title = "Pit Stop Count"),
        alt.Tooltip("constructorName:N", title = "Constructor Name"),
        alt.Tooltip("year:N", title = "Year")
    ]
).add_params(
    constructor_selection,
    year_selection
).transform_filter(
    constructor_selection
).transform_filter(
    year_selection
).properties(
    title = "Distribution of Pit Stop Times",
    width = 350,
    height = 300
)

# Bar chart for constructor vs top 3 finishes
constructor_top_3_bar_chart = alt.Chart(constructor_top_3_per_year_df).mark_bar(
    stroke = "#15151E",
    strokeWidth = 1
).encode(
    alt.X(
        "constructorName:N",
        title = "Constructor",
        sort = "y"
    ),
    alt.Y(
        "Top 3 Finishes:Q",
        title = "Number of Top 3 Finishes"
    ),
    color = alt.condition(
        constructor_selection,
        alt.value("#FF1E00"),
        alt.value("#D3D3D3")
    ),
    tooltip = [
        alt.Tooltip("constructorName:N", title = "Constructor Name"),
        alt.Tooltip("Top 3 Finishes:Q", title = "Top 3 Finishes")
    ]
).add_params(
    constructor_selection,
    year_selection
).transform_filter(
    year_selection
).properties(
    title = "Constructor vs Top 3 Finishes",
    width = 350,
    height = 300
)

# Combine both charts
combined_chart = alt.hconcat(
    pit_stop_histogram,
    constructor_top_3_bar_chart
).resolve_scale(
    color = "independent"
)

# Apply font and size configurations
combined_chart = combined_chart.configure(
    title = {
        "font": "Quicksand, sans-serif",
        "fontSize": 20
    },
    axis = {
        "labelFont": "Quicksand, sans-serif",
        "titleFont": "Quicksand, sans-serif",
        "labelFontSize": 16,
        "titleFontSize": 16
    },
    legend = {
        "labelFont": "Quicksand, sans-serif",
        "titleFont": "Quicksand, sans-serif",
        "labelFontSize": 16,
        "titleFontSize": 16
    }
)

#combined_chart

#combined_chart.save("../../img/pit_stop_histogram_linked_view.html")

C:\Users\DCorc\AppData\Local\Temp\ipykernel_7196\1404304689.py:8: AltairDeprecationWarning: 
Deprecated since `altair=5.0.0`. Use selection_point instead.
  constructor_selection = alt.selection_single(
C:\Users\DCorc\AppData\Local\Temp\ipykernel_7196\1404304689.py:20: AltairDeprecationWarning: 
Deprecated since `altair=5.0.0`. Use selection_point instead.
  year_selection = alt.selection_single(


## Constructor Dominance

In [1]:
import altair as alt
import pandas as pd

constructor_results_df = pd.read_csv("../../data/ergast/constructor_results.csv")
constructors_df = pd.read_csv("../../data/ergast/constructors.csv")
races_df = pd.read_csv("../../data/ergast/races.csv")

# Merge races_df with your existing data
merged_data = pd.merge(constructor_results_df, races_df, on = "raceId")


# Merge with constructors_df to get constructor names
merged_data = pd.merge(merged_data, constructors_df, on = "constructorId")

columns_to_keep = ["constructorId", "points", "year", "name_y"]

merged_data = merged_data[columns_to_keep]

# Rneame columns
merged_data = merged_data.rename(columns = {"name_y": "constructorName"})

# Filter data for years >= 2009
merged_data = merged_data[merged_data["year"] >= 2015]

# Calculate cumulative points by constructor and year
merged_data["cumulative_points"] = merged_data.groupby(["constructorId", "year"])["points"].cumsum()


# Group data by year and constructor name for visualization
data_grouped = merged_data.groupby(["year", "constructorName"], as_index = False).sum()

# Add a rank column based on total points per year
data_grouped["rank"] = data_grouped.groupby("year")["points"].rank(ascending=False, method="dense")

# Convert rank to integers (optional)
data_grouped["rank"] = data_grouped["rank"].astype(int)

# Filter the data to include only ranks 1 through 5
#data_grouped = data_grouped[data_grouped["rank"] <= 5]

#data_grouped = data_grouped[data_grouped["constructorName"].isin(["Mercedes", "Red Bull", "Ferrari", "McLaren"])]

# Map rank to medal labels for legend
data_grouped["rank_label"] = data_grouped["rank"].map({
    1: "1st Place",
    2: "2nd Place",
    3: "3rd Place"
})

# Only keep labels for top 3
data_grouped["rank_label"] = data_grouped["rank_label"].fillna("")

data_grouped.head()

,year,constructorName,constructorId,points,cumulative_points,rank,rank_label
0,2015,Ferrari,114,428.0,4457.0,2,2nd Place
1,2015,Force India,190,136.0,1010.0,5,
2,2015,Lotus F1,3952,78.0,737.0,6,
3,2015,Manor Marussia,3762,0.0,0.0,10,
4,2015,Marussia,824,0.0,0.0,10,


In [2]:
# Ensure Altair allows larger datasets
alt.data_transformers.enable("default", max_rows = None)

# Add an interactive selection for the legend
selection = alt.selection_multi(fields = ["constructorName"], bind = "legend")

# Line Chart
final_positions_line_chart = alt.Chart(data_grouped).mark_line(point = True).encode(
    x = alt.X('year:O', title = 'Year'),  # Use year as x-axis
    y = alt.Y('rank:Q', title = 'Final Position', scale = alt.Scale(reverse = True, domain = [1, 5])),
    color = alt.Color('constructorName:N', title = 'Constructor'),
    tooltip = ['constructorName', 'year', 'rank'],
    opacity = alt.condition(selection, alt.value(1), alt.value(0.1)),
    size = alt.condition(selection, alt.value(3), alt.value(1.5))
).add_selection(
    selection 
).properties(
    title = 'Constructor Final Positions by Year',
    width = 700,
    height = 400
)

#final_positions_line_chart

#final_positions_line_chart.save("../../img/final_positions_line_chart.html")

C:\Users\DCorc\AppData\Local\Temp\ipykernel_23524\3177688272.py:5: AltairDeprecationWarning: 
Deprecated since `altair=5.0.0`. Use selection_point instead.
  selection = alt.selection_multi(fields = ["constructorName"], bind = "legend")
C:\Users\DCorc\AppData\Local\Temp\ipykernel_23524\3177688272.py:15: AltairDeprecationWarning: 
Deprecated since `altair=5.0.0`. Use add_params instead.
  ).add_selection(


In [ ]:
import altair as alt
import pandas as pd

# Selection for the constructor using hover interaction on the heatmap's content
constructor_hover = alt.selection_single(
    fields=['constructorName'],
    empty="none", 
    name="constructor_hover"
)

# Heatmap for rankings
heatmap = alt.Chart(data_grouped).mark_rect().encode(
    x=alt.X("year:O", title="Year"),
    y=alt.Y("constructorName:N", title="Constructor"),
    color=alt.Color("rank_label:N",
                    scale=alt.Scale(
                        domain=["1st Place", "2nd Place", "3rd Place"],
                        range=["#FFB800", "#A9A9A9", "#C87F4D"]
                    ),
                    legend=alt.Legend(title="Podium Position")),
    tooltip=[
        alt.Tooltip("constructorName:N", title="Constructor"),
        alt.Tooltip("rank_label:N", title="Podium Position"),
        alt.Tooltip("year:O", title="Year")
    ]
).properties(
    width=600,
    height=400,
    title="Constructor Final Positions by Year"
).add_selection(
    constructor_hover 
)

# podium positions for the bar chart
podium_positions = ["1st Place", "2nd Place", "3rd Place"]

# Bar chart showing count of finishes for the selected constructor
bar_chart = alt.Chart(data_grouped).mark_bar().encode(
    x=alt.X("rank_label:N", title="Constructors' Championship Final Position",
            scale=alt.Scale(domain=podium_positions),
            axis=alt.Axis(labelAngle=0)), 
    y=alt.Y("count():Q", title="Count of Placements", scale=alt.Scale(domain=[0, 10])),
    color=alt.Color("rank_label:N", legend=None),
    tooltip=[
        alt.Tooltip("rank_label:N", title="Podium Position"),
        alt.Tooltip("count():Q", title="Count of Finishes")
    ]
).transform_filter(
    constructor_hover 
).properties(
    width=600,
    height=250,
    title="Constructors' Championship Finishes for the Selected Constructor"
)

# Combine the heatmap and the bar chart
final_chart = alt.vconcat(heatmap, bar_chart).configure_axis(
    labelFont="Quicksand, sans-serif",
    titleFont="Quicksand, sans-serif",
    labelFontSize=14,
    titleFontSize=16
).configure_title(
    font="Quicksand, sans-serif",
    fontSize=20
).configure_legend(
    labelFont="Quicksand, sans-serif",
    titleFont="Quicksand, sans-serif",
    labelFontSize=14,
    titleFontSize=16
)

#final_chart

#final_chart.save("../../website/img/final_positions_chart.html")

C:\Users\DCorc\AppData\Local\Temp\ipykernel_23524\148136584.py:5: AltairDeprecationWarning: 
Deprecated since `altair=5.0.0`. Use selection_point instead.
  constructor_hover = alt.selection_single(
C:\Users\DCorc\AppData\Local\Temp\ipykernel_23524\148136584.py:30: AltairDeprecationWarning: 
Deprecated since `altair=5.0.0`. Use add_params instead.
  ).add_selection(


alt.VConcatChart(...)